In [1]:
# ================================================================
# CABITZA ET AL. COVID-19 MODEL REPRODUCTION
# EXTERNAL VALIDATION
# LOGISTIC REGRESSION + RANDOM FOREST
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    brier_score_loss
)

warnings.filterwarnings("ignore")


# ================================================================
# 1. SETTINGS
# ================================================================

RANDOM_STATE = 42

PROJECT_ROOT = Path(
    r"C:\Users\bl\Documents\ML-Projects\covid-laboratory-prediction"
)

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "all_training.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "reproduction"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 75)
print("CABITZA ET AL. — EXTERNAL VALIDATION")
print("LOGISTIC REGRESSION + RANDOM FOREST")
print("=" * 75)

print("Dataset exists:", DATA_PATH.exists())


# ================================================================
# 2. LOAD DATA
# ================================================================

df = pd.read_csv(DATA_PATH)

id_col = "Unnamed: 0"
target_col = "target"

ids = df[id_col].astype(str).str.strip()


def assign_cohort(patient_id):

    if patient_id.startswith("A"):
        return "training"

    elif patient_id.startswith("PSMAY"):
        return "training"

    elif patient_id.isdigit() and 2001 <= int(patient_id) <= 2058:
        return "internal_external"

    elif patient_id.isdigit():
        return "external"

    else:
        return "unclassified"


df["cohort"] = ids.apply(
    assign_cohort
)

print("\nReconstructed cohorts:")
print(
    df["cohort"].value_counts()
)


# ================================================================
# 3. DEFINE PREDICTORS
# ================================================================

predictors = [
    col for col in df.columns
    if col not in [
        id_col,
        target_col,
        "cohort"
    ]
]

print(
    "\nNumber of predictors:",
    len(predictors)
)


# ================================================================
# 4. COMBINED OSR + IOG TRAINING DATA
#
# Cabitza external validation:
# models retrained using the combined development and
# internal-external cohorts before evaluation on the
# historical 2018 external cohort.
# ================================================================

combined_training = df[
    df["cohort"].isin(
        [
            "training",
            "internal_external"
        ]
    )
].copy()

X_train = combined_training[
    predictors
].copy()

y_train = combined_training[
    target_col
].astype(int).copy()

print("\nCombined OSR + IOG training cohort")
print("-----------------------------------")

print(
    "N =",
    len(combined_training)
)

print(
    y_train.value_counts().sort_index()
)


# ================================================================
# 5. HISTORICAL 2018 EXTERNAL COHORT
# ================================================================

external = df[
    df["cohort"] == "external"
].copy()

X_external = external[
    predictors
].copy()

y_external = external[
    target_col
].astype(int).copy()

external_ids = external[
    id_col
].astype(str).copy()

print("\n2018 external validation cohort")
print("-----------------------------------")

print(
    "N =",
    len(external)
)

print(
    y_external.value_counts().sort_index()
)

if y_external.nunique() == 1:

    print(
        "\nAll external cases belong to one outcome class."
    )

    print(
        "ROC-AUC and sensitivity are therefore "
        "not estimable."
    )


# ================================================================
# 6. LOGISTIC REGRESSION PIPELINE
#
# Uses the best configuration identified in the
# development reproduction.
# ================================================================

lr_selector = LogisticRegression(
    C=0.1,
    penalty="l1",
    solver="liblinear",
    max_iter=5000,
    random_state=RANDOM_STATE
)

lr_classifier = LogisticRegression(
    C=0.1,
    penalty="l1",
    solver="liblinear",
    max_iter=5000,
    random_state=RANDOM_STATE
)

lr_pipeline = Pipeline([
    (
        "imputer",
        KNNImputer(
            n_neighbors=5
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "rfe",
        RFE(
            estimator=lr_selector,
            n_features_to_select=34,
            step=1
        )
    ),
    (
        "classifier",
        lr_classifier
    )
])


# ================================================================
# 7. RANDOM FOREST PIPELINE
#
# Uses the best configuration identified in the
# development reproduction.
# ================================================================

rf_selector = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_pipeline = Pipeline([
    (
        "imputer",
        KNNImputer(
            n_neighbors=5
        )
    ),
    (
        "rfe",
        RFE(
            estimator=rf_selector,
            n_features_to_select=34,
            step=1
        )
    ),
    (
        "classifier",
        rf_classifier
    )
])


# ================================================================
# 8. TRAIN FINAL LOGISTIC REGRESSION
# ================================================================

print("\n" + "=" * 75)
print("TRAINING FINAL LOGISTIC REGRESSION")
print("=" * 75)

lr_pipeline.fit(
    X_train,
    y_train
)

lr_prob = lr_pipeline.predict_proba(
    X_external
)[:, 1]

lr_pred = (
    lr_prob >= 0.50
).astype(int)


# ================================================================
# 9. TRAIN FINAL RANDOM FOREST
# ================================================================

print("\nTraining final Random Forest...")

rf_pipeline.fit(
    X_train,
    y_train
)

rf_prob = rf_pipeline.predict_proba(
    X_external
)[:, 1]

rf_pred = (
    rf_prob >= 0.50
).astype(int)


# ================================================================
# 10. EXTERNAL VALIDATION METRICS
#
# All external cases are negative.
#
# Therefore:
#
# TN = correctly classified negative cases
# FP = historical cases incorrectly classified COVID+
#
# Specificity = TN / (TN + FP)
#
# Accuracy equals specificity here because every
# external case is negative.
# ================================================================

def external_metrics(
    y_true,
    y_pred,
    y_prob,
    model_name
):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    brier = brier_score_loss(
        y_true,
        y_prob
    )

    return {
        "model": model_name,
        "n_external": len(y_true),
        "true_negative": tn,
        "false_positive": fp,
        "specificity": specificity,
        "accuracy": accuracy,
        "brier_score": brier,
        "mean_predicted_probability": np.mean(
            y_prob
        ),
        "median_predicted_probability": np.median(
            y_prob
        ),
        "max_predicted_probability": np.max(
            y_prob
        )
    }


lr_result = external_metrics(
    y_external,
    lr_pred,
    lr_prob,
    "Logistic Regression"
)

rf_result = external_metrics(
    y_external,
    rf_pred,
    rf_prob,
    "Random Forest"
)


external_results = pd.DataFrame(
    [
        lr_result,
        rf_result
    ]
)


# ================================================================
# 11. DISPLAY EXTERNAL VALIDATION RESULTS
# ================================================================

print("\n" + "=" * 75)
print("EXTERNAL VALIDATION RESULTS")
print("=" * 75)

print(
    external_results.round(4).to_string(
        index=False
    )
)


# ================================================================
# 12. PATIENT-LEVEL EXTERNAL PREDICTIONS
#
# Useful for error analysis.
# No patient identifiers will be committed if this file
# is considered inappropriate for the public repository.
# ================================================================

external_predictions = pd.DataFrame({
    "external_case": np.arange(
        1,
        len(external) + 1
    ),
    "true_target": y_external.to_numpy(),
    "lr_probability": lr_prob,
    "lr_prediction": lr_pred,
    "rf_probability": rf_prob,
    "rf_prediction": rf_pred
})


# ================================================================
# 13. FALSE-POSITIVE ANALYSIS
# ================================================================

lr_false_positive_probabilities = (
    external_predictions.loc[
        external_predictions[
            "lr_prediction"
        ] == 1,
        "lr_probability"
    ]
)

rf_false_positive_probabilities = (
    external_predictions.loc[
        external_predictions[
            "rf_prediction"
        ] == 1,
        "rf_probability"
    ]
)

print("\nFalse-positive counts")
print("---------------------")

print(
    "Logistic Regression:",
    int(np.sum(lr_pred == 1))
)

print(
    "Random Forest:",
    int(np.sum(rf_pred == 1))
)


# ================================================================
# 14. SAVE RESULTS
# ================================================================

external_results.to_csv(
    RESULTS_DIR /
    "lr_rf_external_validation_summary.csv",
    index=False
)

external_predictions.to_csv(
    RESULTS_DIR /
    "lr_rf_external_validation_predictions.csv",
    index=False
)


# ================================================================
# 15. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 75)
print("FINAL EXTERNAL VALIDATION SUMMARY")
print("=" * 75)

for _, row in external_results.iterrows():

    print(
        f"\n{row['model']}"
    )

    print(
        f"Specificity: "
        f"{row['specificity']:.4f}"
    )

    print(
        f"True negatives: "
        f"{int(row['true_negative'])}/"
        f"{int(row['n_external'])}"
    )

    print(
        f"False positives: "
        f"{int(row['false_positive'])}"
    )

    print(
        f"Brier score: "
        f"{row['brier_score']:.4f}"
    )

    print(
        f"Mean predicted COVID probability: "
        f"{row['mean_predicted_probability']:.4f}"
    )

    print(
        f"Maximum predicted COVID probability: "
        f"{row['max_predicted_probability']:.4f}"
    )


print("\nSaved:")
print(
    RESULTS_DIR /
    "lr_rf_external_validation_summary.csv"
)

print(
    RESULTS_DIR /
    "lr_rf_external_validation_predictions.csv"
)

print(
    "\nExternal validation finished."
)

CABITZA ET AL. — EXTERNAL VALIDATION
LOGISTIC REGRESSION + RANDOM FOREST
Dataset exists: True

Reconstructed cohorts:
cohort
training             1624
internal_external      58
external               54
Name: count, dtype: int64

Number of predictors: 34

Combined OSR + IOG training cohort
-----------------------------------
N = 1682
target
0    866
1    816
Name: count, dtype: int64

2018 external validation cohort
-----------------------------------
N = 54
target
0    54
Name: count, dtype: int64

All external cases belong to one outcome class.
ROC-AUC and sensitivity are therefore not estimable.

TRAINING FINAL LOGISTIC REGRESSION

Training final Random Forest...

EXTERNAL VALIDATION RESULTS
              model  n_external  true_negative  false_positive  specificity  accuracy  brier_score  mean_predicted_probability  median_predicted_probability  max_predicted_probability
Logistic Regression          54             49               5       0.9074    0.9074       0.0709              